# Qwen2.5 LoRA 指令微调

本 notebook 包含全部流程：数据加载 → LoRA 训练 → 效果评估 → 量化导出。
在 Colab 中打开后，选择 T4 GPU，从上到下逐格运行即可。

详细原理说明见 README.md。

In [ ]:
# ==================== 配置区 ====================
MODEL_ID = "Qwen/Qwen2.5-1.5B"        # 首次用 1.5B
# MODEL_ID = "Qwen/Qwen2.5-7B"        # 7B 需要 QLoRA，训练更久

MAX_SAMPLES = 3000                     # 训练样本数
BATCH_SIZE = 4                         # 1.5B 用 4；7B 改 2
GRAD_ACCUM = 4                         # 等效 batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS = 2
LR = 2e-4
LORA_RANK = 16
USE_QLORA = "7B" in MODEL_ID
MODEL_TAG = MODEL_ID.split('/')[-1]

print(f"模型: {MODEL_TAG}")
print(f"QLoRA: {USE_QLORA}")
print(f"训练样本: {MAX_SAMPLES}")
print(f"等效 batch: {BATCH_SIZE * GRAD_ACCUM}")

## 1. 安装依赖

In [ ]:
!pip install -q transformers accelerate peft trl datasets
if USE_QLORA:
    !pip install -q bitsandbytes
print("依赖就绪 ✓")

## 2. 数据准备

加载 alpaca-zh 中文指令数据集，统一为 ChatML 格式。

In [ ]:
from datasets import load_dataset

CHAT_TEMPLATE = """<|im_start|>user
{instruction}
<|im_end|>
<|im_start|>assistant
{output}
<|im_end|>"""

dataset = load_dataset("shibing624/alpaca-zh", split="train")
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

def format_chat(ex):
    return {"text": CHAT_TEMPLATE.format(instruction=ex["instruction"], output=ex["output"])}

dataset = dataset.map(format_chat)
print(f"训练数据: {len(dataset)} 条")
print(f"\n示例:\n{dataset[0]['text'][:180]}...")

## 3. 加载基座模型 + LoRA 配置

1.5B 用 FP16 LoRA，7B 用 4-bit QLoRA。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"加载模型: {MODEL_ID}")
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

print(f"✓ 模型加载完成 | {model.num_parameters()/1e9:.2f}B 参数")

## 4. LoRA 训练

训练过程中每 10 步打印一次 loss。关注 loss 的变化趋势。

In [ ]:
import time
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = f"./{MODEL_TAG}_lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, args=training_args,
    train_dataset=dataset, dataset_text_field="text",
    max_seq_length=1024, packing=True,
)

print("开始训练...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

final_loss = trainer.state.log_history[-1].get("train_loss", 0)
print(f"\n✓ 训练完成")
print(f"  耗时: {elapsed/60:.1f} 分钟")
print(f"  最终 loss: {final_loss:.4f}")

## 5. 效果评估

20 道技术问答，对比微调前后的回答质量。

In [ ]:
import json
from peft import PeftModel

# 测试集
test_set = [
    ("Transformer 中的多头注意力机制是如何工作的？", ["多个头", "拼接", "线性变换", "子空间"]),
    ("什么是 KV Cache？在推理中起到什么作用？", ["缓存", "Key", "Value", "加速"]),
    ("LoRA 微调的核心原理是什么？", ["低秩", "冻结", "插入", "参数量"]),
    ("什么是 RAG？它主要解决什么问题？", ["检索", "增强", "生成", "知识库"]),
    ("ReAct 推理范式是什么？", ["推理", "行动", "交替", "工具"]),
    ("AWQ 量化和 GPTQ 量化有什么不同？", ["激活值", "缩放", "INT4", "权重"]),
    ("Python 中装饰器是什么？", ["高阶函数", "增强", "注解", "函数"]),
    ("什么是 Python GIL？", ["全局锁", "线程", "CPython", "并行"]),
    ("FastAPI 和 Flask 的主要区别是什么？", ["异步", "自动文档", "性能", "校验"]),
    ("Docker 中 COPY 和 ADD 指令有什么区别？", ["复制", "URL", "解压", "构建"]),
    ("什么是模型幻觉？有哪些缓解方法？", ["事实错误", "RAG", "微调", "约束"]),
    ("LLM 推理时 temperature 参数的作用是什么？", ["随机性", "采样", "多样性", "控制"]),
    ("什么是 Prompt Engineering？", ["提示", "设计", "Few-shot", "指令"]),
    ("LangChain 中 Chain 和 Agent 的区别？", ["固定", "动态", "工具", "决策"]),
    ("什么是 PagedAttention？", ["分页", "显存", "KV", "碎片"]),
    ("Batch Norm 和 Layer Norm 的区别？", ["维度", "Batch", "特征", "依赖"]),
    ("什么是梯度消失？如何缓解？", ["指数", "残差", "裁剪", "激活"]),
    ("Adam 和 SGD 相比有什么优势？", ["自适应", "动量", "学习率", "收敛"]),
    ("Fine-tuning 和 ICL 有什么区别？", ["权重", "示例", "更新", "泛化"]),
    ("什么是 Tool Calling？", ["结构化", "JSON", "函数", "参数"]),
]

def ask(m, t, q):
    inputs = t(f"<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=96, temperature=0.1, do_sample=False)
    return t.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def hit(answer, kws):
    if not kws: return 0
    return sum(1 for kw in kws if kw.lower() in answer.lower()) / len(kws)

print("加载基座模型...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
).eval()

print("加载微调模型...")
lora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
lora = PeftModel.from_pretrained(lora, OUTPUT_DIR).eval()

base_total, lora_total = 0, 0
better, worse = 0, 0
for i, (q, kws) in enumerate(test_set):
    ba, la = ask(base, tokenizer, q), ask(lora, tokenizer, q)
    bs, ls = hit(ba, kws), hit(la, kws)
    base_total += bs; lora_total += ls
    if ls > bs: better += 1
    elif ls < bs: worse += 1
    if i % 5 == 0:
        d = "↑" if ls > bs else ("↓" if ls < bs else "=")
        print(f"Q{i+1}: [{bs:.0%}→{ls:.0%}] {d}  {q[:40]}...")

n = len(test_set)
print(f"\n{'='*50}")
print(f"基座平均: {base_total/n:.1%}")
print(f"微调平均: {lora_total/n:.1%}")
print(f"提升: {(lora_total-base_total)/base_total*100:+.0f}%")
print(f"更好/更差: {better}/{worse}")
print(f"{'='*50}")

## 6.（可选）保存到 Google Drive

防止 Colab 断开后训练成果丢失。

In [ ]:
from google.colab import drive
import shutil

try:
    drive.mount('/content/drive')
    save_path = f"/content/drive/MyDrive/{MODEL_TAG}_lora"
    shutil.copytree(OUTPUT_DIR, save_path, dirs_exist_ok=True)
    print(f"模型已保存到: {save_path}")
except Exception as e:
    print(f"保存失败: {e}")
    print("模型仍在 Colab 本地，下载前不会丢失。")

## 完成

你已经完成了整个微调流程。接下来可以：
1. 切换 7B 模型再跑一次，看效果差距
2. 用 quantize.py 导出量化模型
3. 把评估结果截图放回 README

详细的原理说明和选型分析见 README.md。